In [2]:
# === SEL 1: SCRAPING ULASAN DARI GOOGLE PLAY STORE ===
# Hapus tanda # di baris bawah jika google-play-scraper belum terpasang:
# !pip install google-play-scraper

from google_play_scraper import reviews, Sort
import pandas as pd, re, time

APPS = {
    "Lapor PAM": "id.co.pamjaya.laporpam.dev",
    "Super App Polri": "superapps.polri.presisi.presisi",
}
PER_SKOR = 400        # maksimum ulasan per bintang per aplikasi (naikkan kalau kurang)

frames = []
for nama, app_id in APPS.items():
    for skor in [1, 2, 3, 4, 5]:
        try:
            hasil, _ = reviews(app_id, lang="id", country="id",
                               sort=Sort.NEWEST, count=PER_SKOR,
                               filter_score_with=skor)
        except Exception as e:
            print("gagal:", nama, skor, e); continue
        d = pd.DataFrame(hasil)
        if d.empty:
            print(nama, "bintang", skor, "-> 0"); continue
        d = d[["content", "score", "at"]].copy()
        d["aplikasi"] = nama
        frames.append(d)
        print(nama, "bintang", skor, "->", len(d))
        time.sleep(1)

mentah = pd.concat(frames, ignore_index=True)
print("Total mentah:", len(mentah))

Lapor PAM bintang 1 -> 108
Lapor PAM bintang 2 -> 4
Lapor PAM bintang 3 -> 4
Lapor PAM bintang 4 -> 1
Lapor PAM bintang 5 -> 100
Super App Polri bintang 1 -> 400
Super App Polri bintang 2 -> 400
Super App Polri bintang 3 -> 400
Super App Polri bintang 4 -> 400
Super App Polri bintang 5 -> 400
Total mentah: 2217


In [3]:
# === SEL 2: BERSIHKAN DATA, LABEL DARI BINTANG, & SIMPAN DATASET FINAL ===
df = mentah.copy()
n_awal = len(df)

# 1. Rapikan teks dan tanggal
df["at"] = pd.to_datetime(df["at"], errors="coerce")
df["content"] = df["content"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# 2. Saring ulasan yang memiliki huruf dan minimal 3 kata
ada_huruf = df["content"].str.contains(r"[A-Za-z]", na=False)
min_3_kata = df["content"].str.split().str.len() >= 3
df = df[ada_huruf & min_3_kata].copy()

# 3. Hapus duplikat ternormalisasi (abaikan perbedaan huruf besar & tanda baca)
df["kunci_dup"] = (
    df["content"]
    .str.lower()
    .str.replace(r"[^a-z0-9 ]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df = df[~df.duplicated(subset=["aplikasi", "kunci_dup"], keep="first")].drop(columns="kunci_dup")

# 4. Hapus ulasan berbahasa Inggris
INGGRIS = re.compile(r"\b(?:the|and|is|this|app|good|nice|very|great|best|worst|bad|useful|helpful|love|not|always|every|time)\b", re.I)
INDO = re.compile(r"\b(?:yang|dan|di|untuk|dengan|pada|ini|dari|dalam|adalah|tidak|gak|ga|nggak|aplikasi|saya|bisa|sudah|udah|banget|sangat|mudah|cepat|memang|terbaik)\b", re.I)
bukan_inggris = ~df["content"].apply(lambda t: len(INGGRIS.findall(t)) >= 2 and len(INDO.findall(t)) <= 1)
df = df[bukan_inggris].reset_index(drop=True)

# 5. Pelabelan otomatis berdasarkan bintang
def label_bintang(skor):
    if skor <= 2:
        return "negatif"
    elif skor == 3:
        return "netral"
    return "positif"

df["id"] = range(1, len(df) + 1)
df["label_sentimen"] = df["score"].apply(label_bintang)

# 6. Susun kolom dan simpan ke 1 file final: ulasan_layanan_publik.csv
kolom_final = ["id", "aplikasi", "at", "content", "score", "label_sentimen"]
df = df[kolom_final]
df.to_csv("ulasan_layanan_publik.csv", index=False, encoding="utf-8-sig")

print(f"Total mentah: {n_awal} -> Total bersih final: {len(df)} ulasan")
print("\nSebaran per aplikasi dan bintang:")
print(df.groupby(["aplikasi", "score"]).size().unstack(fill_value=0))
print("\nSebaran label sentimen final:")
print(df["label_sentimen"].value_counts())
print("\nTersimpan: ulasan_layanan_publik.csv")

Total mentah: 2217 -> Total bersih final: 1619 ulasan

Sebaran per aplikasi dan bintang:
score              1    2    3    4    5
aplikasi                                
Lapor PAM        100    4    4    1   79
Super App Polri  373  372  312  207  167

Sebaran label sentimen final:
label_sentimen
negatif    849
positif    454
netral     316
Name: count, dtype: int64

Tersimpan: ulasan_layanan_publik.csv


In [4]:
print('Jumlah baris:', df.shape[0])
print('Jumlah baris:', df.shape[1])
print('Kolom:', df.columns)

df.head()

Jumlah baris: 1619
Jumlah baris: 6
Kolom: Index(['id', 'aplikasi', 'at', 'content', 'score', 'label_sentimen'], dtype='str')


,id,aplikasi,at,content,score,label_sentimen
0,1,Lapor PAM,2026-10-16 10:09:38,"aplikasi buruk daftar akun aja ga bisa""",1,negatif
1,2,Lapor PAM,2026-10-10 23:46:15,"Apliaksi buruk, Sering force close",1,negatif
2,3,Lapor PAM,2026-10-09 15:59:52,"aplikasi gak berguna! bikin laporan aduan, bel...",1,negatif
3,4,Lapor PAM,2026-10-09 14:30:22,tidak ada kejelasan terkait pencatatan meter air.,1,negatif
4,5,Lapor PAM,2026-10-04 19:45:19,WOOOWWW APLIKASI INI SANGAT MEMBANTU SEKALI UN...,1,negatif


In [5]:
df['label_sentimen'].value_counts()

label_sentimen
negatif    849
positif    454
netral     316
Name: count, dtype: int64